# Exercise: Extend and Evaluate a LangChain RAG Pipeline

In the lesson you built a RAG pipeline with LangChain. You start from the working pipeline and take it further: make its answers **traceable** by returning their sources, **tune** how much context it retrieves, and then **evaluate** the result so you can say which setup is better, not just guess.

For the concepts behind evaluation, see [RAG Concepts](1_rag_concepts.md).

## Learning Objectives

At the end of this notebook, you should be able to:

- Extend a RAG pipeline to return the source passages behind each answer.
- Tune retrieval (the number of chunks) and observe its effect on the answer.
- Evaluate and compare two configurations with an LLM-as-judge (faithfulness and answer relevance).

## Provided: the pipeline from the lesson

Run this cell as is. It gives you `build_chain(pdf_path, k=..., chunk_size=..., chunk_overlap=...)`, which returns a ready retrieval chain, and `ask(question, chain)`, which invokes a chain and returns the full result (the `input`, the retrieved `context`, and the `answer`). It also builds `react_chain` over the React paper for you. You do not need to change anything here.

In [1]:
import warnings

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client
from pypdf import PdfReader

./.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
warnings.filterwarnings("ignore")

In [3]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [4]:
def build_chain(pdf_path, k=4, chunk_size=200, chunk_overlap=50):
    """Load a PDF, build a Chroma retriever, and wire it to the LLM. Returns a retrieval chain."""
    reader = PdfReader(pdf_path)

    docs = [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": pdf_path, "page": i},
        )
        for i, page in enumerate(reader.pages, start=1)
    ]

    chunks = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    ).split_documents(docs)

    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        encode_kwargs={"normalize_embeddings": True},
    )

    store = Chroma.from_documents(
        chunks, embedding, collection_metadata={"hnsw:space": "cosine"}
    )

    retriever = store.as_retriever(search_kwargs={"k": k})

    prompt = Client().pull_prompt(
        "langchain-ai/retrieval-qa-chat",
        include_model=False,
        dangerously_pull_public_prompt=True,
    )

    combine = create_stuff_documents_chain(llm=llm, prompt=prompt)
    return create_retrieval_chain(retriever=retriever, combine_docs_chain=combine)


def ask(question, chain):
    """Invoke a chain and return the full result dict (input, context, answer)."""
    return chain.invoke({"input": question})


# Build the pipeline over the React paper, ready to use
react_chain = build_chain("documents/react_paper.pdf")

## Exercise 1: Make answers traceable with source citations

A RAG answer is only trustworthy if you can check where it came from. The result from `ask` includes a `context` list of the retrieved documents, and each document carries metadata, including its page number in `doc.metadata`.

Write a function that asks a question and returns the answer **together with the source pages** of the chunks that supported it.

Hints:

- Call `ask(question, react_chain)` and read `result["answer"]` and `result["context"]`.
- Each item in `context` is a `Document`; look at `doc.metadata` to find the page.

In [5]:
def answer_with_sources(question):
    """Return the answer plus the source pages of the retrieved chunks."""
    result = ask(question, react_chain)
    pages = sorted({doc.metadata["page"] for doc in result["context"]})
    return {"answer": result["answer"], "pages": pages}

In [6]:
out = answer_with_sources("What is the ReAct framework?")
print(out["answer"])
print("Sources (pages):", out["pages"])

**ReAct** is a framework that lets a large language model (LLM) *think* and *act* in a single, tightly‑coupled loop.  
In a ReAct prompt the model writes a short “thought” (its internal reasoning) and then, on the next line, specifies an “action” (e.g., call an API, look up a fact, or produce a final answer). After the action is executed, the result is fed back into the next reasoning step.  

Key points from the context:

| Feature | What it means |
|---------|---------------|
| **Intuitive & easy to design** | Human annotators simply type their thoughts in natural language on top of the actions. No special syntax or tooling is required. |
| **Alignment & reduced hallucination** | By removing a hallucinating sentence (Act 17) and adding hints (Act 23), ReAct can be nudged to behave more reliably. |
| **Success vs. failure modes** | In the study, ReAct achieved a 14 % success rate versus 6 % for a baseline, but still had a 56 % failure mode. The framework’s design helps identify and mi

## Exercise 2: Tune how much context you retrieve

The retriever returns the top `k` chunks. Too few may miss the answer, too many may dilute the prompt with irrelevant text. `build_chain` accepts a `k` argument, so you can build chains with different values.

Build two chains over the React paper, one with a small `k` and one with a larger `k`, ask both the same question, and compare the answers.

Hint: `chain_small = build_chain("documents/react_paper.pdf", k=2)`.

In [7]:
QUESTION = "How does ReAct combine reasoning and acting, and what benefit does it give on HotpotQA?"
chain_small = build_chain("documents/react_paper.pdf", k=2)
chain_large = build_chain("documents/react_paper.pdf", k=6)

res_small = ask(QUESTION, chain_small)
res_large = ask(QUESTION, chain_large)
print("k=2:", res_small["answer"], "\n")
print("k=6:", res_large["answer"])

k=2: ReAct works by **alternating** between two kinds of steps:

1. **Reasoning steps** – the model writes out a chain of thought, just as a normal language‑model would.
2. **Acting steps** – the model issues an “action” (for example, a query to an external knowledge base) and then receives an “observation” (the result of that query).

By interleaving these reasoning, action, and observation steps, ReAct can **ground its chain of thought in real‑time evidence** rather than relying solely on what is already stored in its parameters.  

On the HotpotQA benchmark, which requires multi‑hop reasoning and often external facts, this grounding gives a measurable boost: ReAct’s ability to fetch and incorporate fresh information leads to higher accuracy (and typically better multi‑hop reasoning scores) compared with models that reason only from their internal knowledge. 

k=6: **How ReAct fuses reasoning and acting**

ReAct is a framework that lets a language model *alternate* between two kinds 

## Exercise 3: Evaluate and compare the two configurations

Comparing answers by eye does not scale. Use the model as a judge to score each answer, then let the scores decide which `k` worked better.

Write a small `llm_judge` function that asks the model to rate something from 1 to 5, then use it to score two things for each answer:

- **Faithfulness**: is the answer supported by the retrieved context?
- **Answer relevance**: does the answer address the question?

Compare the scores for your small-`k` and large-`k` chains, and write a sentence on which you would choose and why.

Hint: build the judging prompt from the question, the context, and the answer, send it with the provided `llm`, and parse a single digit from the reply.

In [8]:
import re


def llm_judge(instruction):
    """Ask the model to rate something from 1 to 5 and return the number."""
    reply = llm.invoke(
        instruction + "\n\nReply with a single integer from 1 to 5 and nothing else."
    ).content
    match = re.search(r"[1-5]", reply or "")
    return int(match.group()) if match else None

In [9]:
def score(result):
    context = "\n---\n".join(d.page_content for d in result["context"])
    faith = llm_judge(
        "Rate how well the ANSWER is supported by the CONTEXT "
        f"(5 = fully supported).\n\nCONTEXT:\n{context}\n\nANSWER:\n{result['answer']}"
    )
    relevance = llm_judge(
        "Rate how well the ANSWER addresses the QUESTION "
        f"(5 = fully addresses it).\n\nQUESTION: {result['input']}\n\nANSWER:\n{result['answer']}"
    )
    return {"faithfulness": faith, "relevance": relevance}


print("k=2:", score(res_small))
print("k=6:", score(res_large))

k=2: {'faithfulness': 3, 'relevance': 5}


k=6: {'faithfulness': 5, 'relevance': 5}


## Conclusion and Reflection

- Did citations change how much you trust the answers? When would source pages matter most?
- Which `k` gave better judge scores, and did that match your own reading of the answers?
- The judge is itself an LLM and can be biased. How would you make the evaluation more reliable, and where would a tool like RAGAS or a labelled test set help?

### My answers

- **Citations:** returning the source pages (for example pages 4, 6 and 14) lets me verify a claim in the paper instead of trusting the model, which matters most for legal, medical or any high-stakes use.
- **Which `k`:** the judge scored `k=2` at faithfulness 3 / relevance 5 and `k=6` at faithfulness 5 / relevance 5. I would choose `k=6`: with chunks of only 200 characters, two chunks are too little context, and the small-`k` answer contained claims (such as the HotpotQA gain) that the retrieved text did not fully support. Reading the answers myself agreed with this: the `k=6` answer was better grounded, although longer.
- **Reliability of the judge:** the judge is an LLM with its own biases (for example favouring longer answers) and one run is noisy. I would run it several times and average, use a different model as judge, and add a small labelled test set with reference answers; RAGAS provides standard faithfulness and relevancy metrics for exactly this.
